# Merge of the four NetFlow datasets — Phase 3

**Owner:** Ahmed Hamouda. Stacks the four cleaned files (same 56 columns, same order) and applies, once, the approved
after-merge decisions (`report/data_problems/phase2_data_problems.pdf`, section 5.1).

The merge is a **stack** (`pd.concat`), not a join: the datasets contain different flows from different networks, so there
is no key to match on. The columns are lined up by name; `source_dataset` tags every row with its network.

| Step | What | Decision |
|---|---|---|
| 1 | Load the four cleaned files and check columns and types | – |
| 2 | Stack them; add `network_type` | – |
| 3 | ICMP fields → 0 outside ICMP (1) and ICMPv6 (58) | M6 |
| 4 | DNS TTL above 2^31−1 → 0 (RFC 2181), then merge the few rows that became identical | M9 |
| 5 | Drop the copy columns `MIN/MAX_IP_PKT_LEN`, `TCP_FLAGS`, `ICMP_IPV4_TYPE` | M7, M8 |
| 6 | Flag behaviours that have a different family in another dataset | M11 |
| 7 | Code columns → categorical; timestamps → datetime | M1, M2 |
| 8 | Checks and save | – |

**Not done here:** the train/test split, the class / network weights and the realistic 95/5 test sets (section 6) are
learned from data and come after the split. Columns kept but never used as model features are listed in `NOT_FEATURES`
(`src/config.py`).

## Step 1 — Load the four cleaned files and check them

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from src.config import *

pd.set_option("display.max_columns", 100)
SOURCES = ["UNSW", "CIC18", "ToN", "BoT"]
files = {s: PROCESSED_DIR / f"{s}_clean.parquet" for s in SOURCES}

schemas = {s: pq.read_schema(f) for s, f in files.items()}
names = {s: schemas[s].names for s in SOURCES}
print("same columns and order in all four:", all(names[s] == names["UNSW"] for s in SOURCES))
types = pd.DataFrame({s: {f.name: str(f.type) for f in schemas[s]} for s in SOURCES})
print("columns whose stored type differs between files (pandas upcasts them when stacking):")
display(types[types.nunique(axis=1) > 1])

parts = {s: pd.read_parquet(f) for s, f in files.items()}
size = pd.DataFrame({"rows": {s: len(p) for s, p in parts.items()},
                     "flows represented": {s: int(p["count"].sum()) for s, p in parts.items()}})
size.loc["total"] = size.sum()
size

same columns and order in all four: True
columns whose stored type differs between files (pandas upcasts them when stacking):


,UNSW,CIC18,ToN,BoT
DNS_TTL_ANSWER,int64,int32,int32,int64


,rows,flows represented
UNSW,1405100,2350609
CIC18,8945792,19285037
ToN,5637642,24442770
BoT,1563315,16933808
total,17551849,63012224


## Step 2 — Stack the four files and add `network_type`
`network_type` comes from `DATASETS` in `src/config.py`. Like `source_dataset`, it is for the analysis and never a model
feature.

In [2]:
df = pd.concat([parts[s] for s in SOURCES], ignore_index=True)
del parts
df["network_type"] = df["source_dataset"].map({s: d["network"] for s, d in DATASETS.items()})
flows_total = int(df["count"].sum())
steps = []
def log(step, decision, what):
    steps.append({"step": step, "decision": decision, "rows": len(df), "columns": df.shape[1], "what": what})
    print(f"{step}: {len(df):,} rows x {df.shape[1]} columns  ({what})")
log("2 stack", "-", "four files stacked, network_type added")
print("rows per family and dataset:")
pd.crosstab(df["attack_family"], df["source_dataset"], margins=True)

2 stack: 17,551,849 rows x 57 columns  (four files stacked, network_type added)
rows per family and dataset:


source_dataset,BoT,CIC18,ToN,UNSW,All
attack_family,,,,,
Benign,21951,8178232,2282642,1307351,11790176
Credential attacks,0,94890,697124,0,792014
Data theft,448,0,0,0,448
Denial of service,642187,642600,573277,5360,1863424
Exploitation and intrusion,0,27213,0,42740,69953
Fuzzing and crypto,0,0,0,32713,32713
Interception and spoofing,0,0,4951,0,4951
Malware and botnets,0,2804,23471,3802,30077
Reconnaissance,898729,0,225074,13134,1136937


## Step 3 — ICMP fields → 0 outside ICMP and ICMPv6 (M6)
`ICMP_TYPE` only has a meaning for protocol 1 (ICMP) and 58 (ICMPv6); on TCP and UDP flows its values are leftovers of the
exporter. (`ICMP_IPV4_TYPE` is dropped in step 5.)

In [3]:
icmp_proto = df["PROTOCOL"].isin([1, 58])
wrong = ~icmp_proto & (df["ICMP_TYPE"] != 0)
print("rows changed per dataset:")
display(wrong.groupby(df["source_dataset"]).sum().rename("rows with an ICMP value outside ICMP"))
df.loc[wrong, ["ICMP_TYPE", "ICMP_IPV4_TYPE"]] = 0
log("3 ICMP fields", "M6", f"{int(wrong.sum()):,} rows set to 0")

rows changed per dataset:


source_dataset
BoT         8063
CIC18    1384904
ToN       193652
UNSW     1293347
Name: rows with an ICMP value outside ICMP, dtype: int64

3 ICMP fields: 17,551,849 rows x 57 columns  (2,879,966 rows set to 0)


## Step 4 — Invalid DNS TTL → 0 (M9), then merge rows that became identical
RFC 2181: a DNS TTL above 2^31−1 seconds is invalid and must be read as 0. After the change, a few rows can become identical
to another row of the same dataset and label; they are merged and their `count` added up, so there is still one row per
behaviour.

In [4]:
bad_ttl = df["DNS_TTL_ANSWER"] > 2**31 - 1
display(bad_ttl.groupby(df["source_dataset"]).sum().rename("rows with an invalid DNS TTL"))
df.loc[bad_ttl, "DNS_TTL_ANSWER"] = 0
log("4a DNS TTL", "M9", f"{int(bad_ttl.sum()):,} values set to 0")

not_behaviour = NETFLOW_NOT_BEHAVIOUR + META_COLUMNS
h = behaviour_hash(df, not_behaviour)
key = pd.DataFrame({"src": df["source_dataset"].to_numpy(), "h": h, "y": df["Attack"].to_numpy()})
first = ~key.duplicated(["src", "h", "y"]).to_numpy()
total = key.assign(c=df["count"].to_numpy()).groupby(["src", "h", "y"], sort=False)["c"].transform("sum").to_numpy()
rows_before = len(df)
df["count"] = total
df = df[first].reset_index(drop=True)
h = h[first]
del key, total, first
assert int(df["count"].sum()) == flows_total
log("4b re-collapse", "section 6", f"{rows_before - len(df):,} rows merged into others")

source_dataset
BoT       2
CIC18     0
ToN       0
UNSW     12
Name: rows with an invalid DNS TTL, dtype: int64

4a DNS TTL: 17,551,849 rows x 57 columns  (14 values set to 0)


4b re-collapse: 17,551,848 rows x 57 columns  (1 rows merged into others)


## Step 5 — Drop the copy columns (M7, M8)
`MIN_IP_PKT_LEN` is 0 on one-way flows (impossible) and otherwise equals `SHORTEST_FLOW_PKT`; `MAX_IP_PKT_LEN` always equals
`LONGEST_FLOW_PKT`; `TCP_FLAGS` = client OR server flags; `ICMP_IPV4_TYPE` = `ICMP_TYPE` ÷ 256. Nothing is lost.

In [5]:
copies = ["MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN", "TCP_FLAGS", "ICMP_IPV4_TYPE"]
assert (df["MAX_IP_PKT_LEN"] == df["LONGEST_FLOW_PKT"]).all()
assert (df["TCP_FLAGS"] == (df["CLIENT_TCP_FLAGS"] | df["SERVER_TCP_FLAGS"])).all()
df = df.drop(columns=copies)
log("5 copy columns", "M7, M8", "4 columns dropped")

5 copy columns: 17,551,848 rows x 53 columns  (4 columns dropped)


## Step 6 — Flag behaviours with a different family in another dataset (M11)
Each dataset alone is consistent, but the same behaviour can be, for example, Reconnaissance in BoT-IoT and Benign in
ToN-IoT. Those rows get `ambiguous_across_sources = True`. **No row is removed**: models are trained with and without them,
and the split keeps each behaviour on one side only.

In [6]:
g = pd.DataFrame({"h": h, "src": df["source_dataset"].to_numpy(), "fam": df["attack_family"].to_numpy()})
per_h = g.groupby("h").agg(sources=("src", "nunique"), families=("fam", "nunique"))
amb_h = per_h.index[(per_h["sources"] > 1) & (per_h["families"] > 1)]
df["ambiguous_across_sources"] = np.isin(h, amb_h)
del g, h
print(f"behaviours with a different family in another dataset: {len(amb_h):,}")
display(df.groupby("source_dataset")[["ambiguous_across_sources"]].sum()
        .join(df[df["ambiguous_across_sources"]].groupby("source_dataset")["count"].sum().rename("flows represented")))
display(pd.crosstab(df.loc[df["ambiguous_across_sources"], "attack_family"], df.loc[df["ambiguous_across_sources"], "source_dataset"]))
log("6 ambiguous flag", "M11", f"{int(df['ambiguous_across_sources'].sum()):,} rows flagged, none removed")

behaviours with a different family in another dataset: 17,587


,ambiguous_across_sources,flows represented
source_dataset,,
BoT,15696,83986.0
CIC18,2203,133484.0
ToN,20554,5346962.0
UNSW,0,NaN


source_dataset,BoT,CIC18,ToN
attack_family,,,
Benign,23,1864,17402
Credential attacks,0,0,49
Data theft,2,0,0
Denial of service,7,0,395
Exploitation and intrusion,0,339,0
Interception and spoofing,0,0,9
Malware and botnets,0,0,49
Reconnaissance,15664,0,2641
Web and injection,0,0,9


6 ambiguous flag: 17,551,848 rows x 54 columns  (38,453 rows flagged, none removed)


## Step 7 — Code columns → categorical (M1), timestamps → datetime (M2)
Ports, protocol numbers, flags and ICMP / DNS / FTP codes are identifiers, not quantities: an average port has no meaning.
The encoding for the model (e.g. port ranges) is chosen in feature engineering. The timestamps become readable dates for
plots over time and a time-based split; they are never model features (`NOT_FEATURES`).

In [7]:
for c in CODE_COLUMNS:
    df[c] = df[c].astype("category")
for c in ["FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS"]:
    df[c] = pd.to_datetime(df[c], unit="ms")
for c in ["Attack", "source_dataset", "network_type", "attack_label", "attack_family"]:
    df[c] = df[c].astype("category")
print("capture period per dataset:")
display(df.groupby("source_dataset", observed=True)["FLOW_START_MILLISECONDS"].agg(["min", "max"]))
log("7 types", "M1, M2", f"{len(CODE_COLUMNS)} code columns categorical, 2 timestamps datetime")
df.dtypes.astype(str).value_counts()

capture period per dataset:


,min,max
source_dataset,,
BoT,2018-05-15 00:27:11.665,2018-06-19 04:15:36.953
CIC18,2018-02-14 12:28:07.705,2018-03-02 20:04:25.524
ToN,2019-04-23 14:09:50.618,2019-04-29 14:45:50.918
UNSW,2015-01-22 11:49:41.000,2015-02-18 12:29:24.917


7 types: 17,551,848 rows x 54 columns  (10 code columns categorical, 2 timestamps datetime)


int32             34
category          15
datetime64[ms]     2
int64              1
int8               1
bool               1
Name: count, dtype: int64

## Step 8 — Checks and save

In [8]:
checks = validate(df)
print("validate(): total violations", int(checks["violations"].sum()))
num = df.select_dtypes("number")
print("NaN cells:", int(df.isna().sum().sum()), "| inf cells:", int(np.isinf(num).sum().sum()))
print("flows represented:", f"{int(df['count'].sum()):,}", "(before the merge:", f"{flows_total:,})")
print("columns:", df.shape[1], "| model feature candidates:", len([c for c in df.columns if c not in NOT_FEATURES]))
summary = df.groupby(["source_dataset", "attack_family", "attack_label"], observed=True).agg(
    rows=("count", "size"), flows=("count", "sum"), ambiguous_rows=("ambiguous_across_sources", "sum"))
summary

validate(): total violations 0


NaN cells: 0 | inf cells: 0
flows represented: 63,012,224 (before the merge: 63,012,224)
columns: 54 | model feature candidates: 39


rows     flows  \
source_dataset attack_family              attack_label                         
BoT            Benign                     Benign             21951     51989   
               Data theft                 Theft                448      1615   
               Denial of service          DDoS              287453   7150882   
                                          DoS               354734   8034190   
               Reconnaissance             Reconnaissance    898728   1695132   
CIC18          Benign                     Benign           8178232  17392754   
               Credential attacks         FTP brute force        1    193360   
                                          SSH brute force    93979     94207   
                                          Web brute force      910       948   
               Denial of service          DDoS HOIC         244060   1032311   
                                          DDoS LOIC-HTTP    265826    280264   
                                          DDoS LOIC-UDP       1723      1725   
                                          DoS GoldenEye      27410     30650   
                                          DoS Hulk          100071    100075   
                                          DoS Slowloris       3510     18020   
               Exploitation and intrusion Infiltration       27213     29209   
               Malware and botnets        Bot                 2804    111460   
               Web and injection          SQL injection         53        54   
ToN            Benign                     Benign           2282642  15078514   
               Credential attacks         Password          697124   1594777   
               Denial of service          DDoS              451474   4141256   
                                          DoS               121803    203456   
               Interception and spoofing  MITM                4951      5268   
               Malware and botnets        Backdoor           20641    101692   
                                          Ransomware          2830      3971   
               Reconnaissance             Scanning          225074   1358977   
               Web and injection          Injection         333652    365308   
                                          XSS              1497451   1589551   
UNSW           Benign                     Benign           1307351   2222930   
               Denial of service          DoS                 5360      5971   
               Exploitation and intrusion Exploits           40424     42744   
                                          Shellcode           2316      2381   
               Fuzzing and crypto         Fuzzers            26987     33816   
                                          Generic             5726     19651   
               Malware and botnets        Backdoor            3656      4658   
                                          Worms                146       158   
               Reconnaissance             Analysis            1226      1226   
                                          Reconnaissance     11908     17074   

                                                           ambiguous_rows  
source_dataset attack_family              attack_label                     
BoT            Benign                     Benign                       23  
               Data theft                 Theft                         2  
               Denial of service          DDoS                          5  
                                          DoS                           2  
               Reconnaissance             Reconnaissance            15664  
CIC18          Benign                     Benign                     1864  
               Credential attacks         FTP brute force               0  
                                          SSH brute force               0  
                                          Web brute force               0  
               Denial of 

In [9]:
log("8 final", "-", "merged table")
pd.DataFrame(steps).to_csv(TABLE_DIR / "merged_quality.csv", index=False)
summary.to_csv(TABLE_DIR / "merged_labels.csv")
out = PROCESSED_DIR / "netflow_merged.parquet"
df.to_parquet(out, index=False)
print("saved", out.relative_to(ROOT), f"({out.stat().st_size / 1e6:.0f} MB)")
display(pd.DataFrame(steps))
pd.crosstab(df["attack_family"], df["source_dataset"], margins=True)

8 final: 17,551,848 rows x 54 columns  (merged table)


saved data\processed\netflow_merged.parquet (780 MB)


,step,decision,rows,columns,what
0,2 stack,-,17551849,57,"four files stacked, network_type added"
1,3 ICMP fields,M6,17551849,57,"2,879,966 rows set to 0"
2,4a DNS TTL,M9,17551849,57,14 values set to 0
3,4b re-collapse,section 6,17551848,57,1 rows merged into others
4,5 copy columns,"M7, M8",17551848,53,4 columns dropped
5,6 ambiguous flag,M11,17551848,54,"38,453 rows flagged, none removed"
6,7 types,"M1, M2",17551848,54,"10 code columns categorical, 2 timestamps date..."
7,8 final,-,17551848,54,merged table


source_dataset,BoT,CIC18,ToN,UNSW,All
attack_family,,,,,
Benign,21951,8178232,2282642,1307351,11790176
Credential attacks,0,94890,697124,0,792014
Data theft,448,0,0,0,448
Denial of service,642187,642600,573277,5360,1863424
Exploitation and intrusion,0,27213,0,42740,69953
Fuzzing and crypto,0,0,0,32713,32713
Interception and spoofing,0,0,4951,0,4951
Malware and botnets,0,2804,23471,3802,30077
Reconnaissance,898728,0,225074,13134,1136936
